#  Resume AI Feedback — Gemini 2.5 Flash

**Author: Muhammad Nouman Qaiser**

This notebook analyzes a resume PDF and returns:
- **Content Score** — writing quality, action verbs, achievements
- **Structure Score** — flow, section ordering, readability
- **ATS Score** — keyword density, parseable format, standard headings
- **Overall Score** — weighted average
- **Actionable suggestions** to improve the resume

---
### How to use
1. Place your resume PDF in the **same folder** as this notebook
2. Set your `GEMINI_API_KEY` in **Cell 2**
3. Set your PDF filename in **Cell 2**
4. Run all cells: **Kernel → Restart & Run All**

In [11]:
# Cell 1 — Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'google-generativeai', 'pypdf', 'python-dotenv', '--quiet'])
print(' Dependencies ready')

 Dependencies ready


In [16]:
# Cell 2 — CONFIGURATION  ← Edit this cell

# Your Gemini API key (get one free at https://aistudio.google.com/apikey)
GEMINI_API_KEY = "your_gemini_api_key_here"

# Name of your resume PDF (must be in the same folder as this notebook)
RESUME_FILENAME = "MUHAMMAD-NOUMAN-QAISER-.pdf"

print(f' Resume file : {RESUME_FILENAME}')
print(f' API key set : {"Yes" if GEMINI_API_KEY != "your_gemini_api_key_here" else "❌ NOT SET — edit this cell"}')

 Resume file : MUHAMMAD-NOUMAN-QAISER-.pdf
 API key set : ❌ NOT SET — edit this cell


In [13]:
# Cell 3 — Extract text from the PDF
import os
from pypdf import PdfReader

# Look for the PDF in the same folder as the notebook
notebook_dir = os.getcwd()
pdf_path = os.path.join(notebook_dir, RESUME_FILENAME)

if not os.path.exists(pdf_path):
    raise FileNotFoundError(
        f"Could not find '{RESUME_FILENAME}' in {notebook_dir}\n"
        f"Make sure the PDF is in the same folder as this notebook."
    )

reader = PdfReader(pdf_path)
pages = reader.pages
resume_text = ''

for i, page in enumerate(pages):
    text = page.extract_text() or ''
    resume_text += text + '\n'

resume_text = resume_text.strip()
word_count = len(resume_text.split())
file_size_kb = os.path.getsize(pdf_path) / 1024

print(f' PDF loaded successfully')
print(f'   File     : {RESUME_FILENAME} ({file_size_kb:.1f} KB)')
print(f'   Pages    : {len(pages)}')
print(f'   Words    : {word_count}')
print(f'   Chars    : {len(resume_text)}')

if word_count < 20:
    print('\n  Very little text extracted.')
    print('   This may be a scanned/image-based PDF.')
    print('   Try: Open in Chrome → File → Print → Save as PDF')
else:
    print(f'\n--- Text preview (first 400 chars) ---')
    print(resume_text[:400])
    print('...')

 PDF loaded successfully
   File     : MUHAMMAD-NOUMAN-QAISER-.pdf (68.7 KB)
   Pages    : 5
   Words    : 1198
   Chars    : 8560

--- Text preview (first 400 chars) ---
MUHAMMAD NOUMAN QAISER
15 Northam Avenue, Highton Geelong

 
+61-466 126 817
 

 
m.noumanqaiser.mughal@gmail.com
 

 
NOUMAN QAISER
MUGHAL
DEAKIN UNIVERSITY (GEELONG, AUSTRALIA )
1st July 2024
 
- 
Continue
UNUVERSITY OF SIALKOT (SIALKOT, PAKISTAN)
May 2019
 
- 
September 2023
Admission Office University of Sialkot
March 1,2020
 
- 
October 2020
Grand Asian University Sialkot
April 2024
 
- 
J
...


In [8]:
# Cell 4 — Send to Gemini 2.5 Flash for analysis
import google.generativeai as genai
import json
import re
import time

genai.configure(api_key=GEMINI_API_KEY)

# max_output_tokens raised to 8192 to prevent truncated JSON
model = genai.GenerativeModel(
    model_name='gemini-2.5-flash',
    generation_config=genai.GenerationConfig(
        response_mime_type='application/json',
        temperature=0.2,
        max_output_tokens=8192,
    ),
    system_instruction=(
        'You are an expert resume coach and ATS specialist with 15+ years of HR experience. '
        'Evaluate resumes objectively and return ONLY a valid JSON object — no markdown, no backticks.'
    )
)

prompt = f"""Analyze the following resume and return a JSON object with EXACTLY this schema:

{{
  "scores": {{
    "content": <integer 0-100>,
    "structure": <integer 0-100>,
    "ats": <integer 0-100>,
    "overall": <integer 0-100>
  }},
  "summary": "<2-3 sentence overview of the resume's current state>",
  "strengths": ["<strength 1>", "<strength 2>", "<strength 3>"],
  "weaknesses": ["<weakness 1>", "<weakness 2>", "<weakness 3>"],
  "suggestions": [
    {{
      "category": "<Content|Structure|ATS|Keywords|Formatting>",
      "priority": "<High|Medium|Low>",
      "suggestion": "<specific actionable improvement>",
      "impact": "<why this matters for the job search>"
    }}
  ],
  "keywords": {{
    "found": ["<keyword present in resume>"],
    "missing": ["<important missing keyword>"]
  }},
  "sectionAnalysis": {{
    "hasContactInfo": <true|false>,
    "hasSummary": <true|false>,
    "hasExperience": <true|false>,
    "hasEducation": <true|false>,
    "hasSkills": <true|false>,
    "hasAchievements": <true|false>
  }}
}}

Scoring rubric:
- content (0-100): Writing quality, action verbs, quantified achievements, relevance
- structure (0-100): Logical flow, section ordering, readability, consistency
- ats (0-100): Keyword density, standard headings, parseable format, no tables/graphics
- overall (0-100): Weighted — content 40% + structure 30% + ats 30%

Keep suggestions concise (1-2 sentences each). Provide exactly 5 suggestions ordered by priority.

Resume ({len(pages)} pages, {word_count} words):
---
{resume_text[:8000]}
---"""

print(' Sending to Gemini 2.5 Flash...')
start = time.time()

response = model.generate_content(prompt)
raw = response.text

elapsed = time.time() - start
print(f' Response received in {elapsed:.1f}s')
print(f'   Raw response length: {len(raw)} chars')

# Robust JSON extraction — handles markdown fences, leading/trailing text
def extract_json(text):
    # Try direct parse first
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    # Strip markdown fences
    text = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.MULTILINE)
    text = re.sub(r'```\s*$', '', text.strip(), flags=re.MULTILINE)
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    # Extract first complete JSON object using brace matching
    start_idx = text.find('{')
    if start_idx == -1:
        raise ValueError('No JSON object found in response')
    depth = 0
    in_string = False
    escape_next = False
    for i, ch in enumerate(text[start_idx:], start=start_idx):
        if escape_next:
            escape_next = False
            continue
        if ch == '\\' and in_string:
            escape_next = True
            continue
        if ch == '"':
            in_string = not in_string
        if not in_string:
            if ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    return json.loads(text[start_idx:i+1])
    raise ValueError(f'Could not extract valid JSON. Response preview: {text[:300]}')

try:
    feedback = extract_json(raw)
except Exception as e:
    print(f'❌ JSON parse error: {e}')
    print(f'Raw response (first 500 chars):\n{raw[:500]}')
    raise

# Clamp scores 0-100
for key in ['content', 'structure', 'ats', 'overall']:
    if key in feedback.get('scores', {}):
        feedback['scores'][key] = min(100, max(0, round(feedback['scores'][key])))

print('📊 Analysis complete!')

🤖 Sending to Gemini 2.5 Flash...
✅ Response received in 17.5s
   Raw response length: 4338 chars
📊 Analysis complete!


In [9]:
# Cell 5 — Display results
from IPython.display import display, HTML

def score_color(n):
    if n >= 75: return '#16a34a'   # green
    if n >= 50: return '#d97706'   # amber
    return '#dc2626'               # red

def score_label(n):
    if n >= 80: return 'Excellent'
    if n >= 65: return 'Good'
    if n >= 50: return 'Fair'
    return 'Needs Work'

scores = feedback['scores']

html = f"""
<style>
  .rf {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 780px; color: #111; }}
  .rf h2 {{ font-size: 20px; font-weight: 600; margin: 24px 0 12px; border-bottom: 1px solid #e5e7eb; padding-bottom: 6px; }}
  .rf h3 {{ font-size: 15px; font-weight: 600; margin: 0 0 4px; }}
  .score-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 16px 0; }}
  .score-card {{ background: #f9fafb; border: 1px solid #e5e7eb; border-radius: 10px; padding: 16px; text-align: center; }}
  .score-num {{ font-size: 32px; font-weight: 700; }}
  .score-lbl {{ font-size: 12px; color: #6b7280; text-transform: uppercase; letter-spacing: .05em; margin-bottom: 4px; }}
  .score-tag {{ font-size: 11px; font-weight: 500; margin-top: 4px; }}
  .summary {{ background: #eff6ff; border-left: 4px solid #3b82f6; border-radius: 4px; padding: 14px 16px; margin: 12px 0; font-size: 14px; line-height: 1.6; color: #1e3a8a; }}
  .two-col {{ display: grid; grid-template-columns: 1fr 1fr; gap: 16px; margin: 12px 0; }}
  .card {{ border: 1px solid #e5e7eb; border-radius: 10px; padding: 16px; }}
  .card ul {{ margin: 0; padding-left: 18px; }}
  .card li {{ font-size: 13px; line-height: 1.7; color: #374151; }}
  .suggestion {{ border: 1px solid #e5e7eb; border-radius: 8px; padding: 14px 16px; margin: 10px 0; }}
  .s-header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px; }}
  .badge {{ font-size: 11px; font-weight: 600; padding: 3px 10px; border-radius: 20px; }}
  .high {{ background: #fee2e2; color: #991b1b; }}
  .medium {{ background: #fef3c7; color: #92400e; }}
  .low {{ background: #dcfce7; color: #166534; }}
  .cat-badge {{ font-size: 11px; padding: 2px 8px; border-radius: 4px; background: #e5e7eb; color: #374151; }}
  .s-text {{ font-size: 13px; color: #111; margin: 4px 0; }}
  .s-impact {{ font-size: 12px; color: #6b7280; font-style: italic; }}
  .kw-grid {{ display: flex; flex-wrap: wrap; gap: 6px; margin-top: 8px; }}
  .kw {{ font-size: 12px; padding: 3px 10px; border-radius: 20px; }}
  .kw-found {{ background: #dcfce7; color: #166534; }}
  .kw-missing {{ background: #fee2e2; color: #991b1b; }}
  .section-grid {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; margin-top: 8px; }}
  .section-item {{ display: flex; align-items: center; gap: 6px; font-size: 13px; }}
</style>

<div class='rf'>
  <h2>📊 Scores</h2>
  <div class='score-grid'>
"""

score_labels = {'content': 'Content', 'structure': 'Structure', 'ats': 'ATS', 'overall': 'Overall'}
for key, label in score_labels.items():
    n = scores.get(key, 0)
    color = score_color(n)
    html += f"""
    <div class='score-card'>
      <div class='score-lbl'>{label}</div>
      <div class='score-num' style='color:{color}'>{n}</div>
      <div class='score-tag' style='color:{color}'>{score_label(n)}</div>
    </div>"""

html += f"""
  </div>

  <h2>📝 Summary</h2>
  <div class='summary'>{feedback.get('summary', '')}</div>

  <h2>💪 Strengths &amp; Weaknesses</h2>
  <div class='two-col'>
    <div class='card'>
      <h3 style='color:#16a34a'>✅ Strengths</h3>
      <ul>{''.join(f"<li>{s}</li>" for s in feedback.get('strengths', []))}</ul>
    </div>
    <div class='card'>
      <h3 style='color:#dc2626'>⚠️ Weaknesses</h3>
      <ul>{''.join(f"<li>{w}</li>" for w in feedback.get('weaknesses', []))}</ul>
    </div>
  </div>

  <h2>🎯 Suggestions</h2>
"""

for s in feedback.get('suggestions', []):
    priority = s.get('priority', 'Medium').lower()
    html += f"""
  <div class='suggestion'>
    <div class='s-header'>
      <span class='cat-badge'>{s.get('category', '')}</span>
      <span class='badge {priority}'>{s.get('priority', '')}</span>
    </div>
    <div class='s-text'>🔹 {s.get('suggestion', '')}</div>
    <div class='s-impact'>💡 {s.get('impact', '')}</div>
  </div>"""

# Keywords
kw = feedback.get('keywords', {})
found_kw = ''.join(f"<span class='kw kw-found'>{k}</span>" for k in kw.get('found', []))
missing_kw = ''.join(f"<span class='kw kw-missing'>{k}</span>" for k in kw.get('missing', []))

html += f"""
  <h2>🔑 Keywords</h2>
  <div class='two-col'>
    <div class='card'>
      <h3 style='color:#16a34a'>Found</h3>
      <div class='kw-grid'>{found_kw}</div>
    </div>
    <div class='card'>
      <h3 style='color:#dc2626'>Missing</h3>
      <div class='kw-grid'>{missing_kw}</div>
    </div>
  </div>
"""

# Section analysis
sa = feedback.get('sectionAnalysis', {})
section_map = [
    ('hasContactInfo', 'Contact Info'),
    ('hasSummary', 'Summary'),
    ('hasExperience', 'Experience'),
    ('hasEducation', 'Education'),
    ('hasSkills', 'Skills'),
    ('hasAchievements', 'Achievements'),
]

html += "<h2>📋 Section Checklist</h2><div class='section-grid'>"
for key, label in section_map:
    present = sa.get(key, False)
    icon = '✅' if present else '❌'
    color = '#16a34a' if present else '#dc2626'
    html += f"<div class='section-item'><span>{icon}</span><span style='color:{color}'>{label}</span></div>"

html += f"""
  </div>
  <p style='color:#9ca3af; font-size:12px; margin-top:24px;'>Analysis by Gemini 2.5 Flash · Processed in {elapsed:.1f}s · {word_count} words analyzed</p>
</div>
"""

display(HTML(html))

In [14]:
# Cell 6 — Save raw JSON output to file (optional)
import json

output_path = os.path.join(notebook_dir, 'resume_feedback_output.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump({
        'meta': {
            'filename': RESUME_FILENAME,
            'pages': len(pages),
            'wordCount': word_count,
            'processingTimeSeconds': round(elapsed, 2)
        },
        'feedback': feedback
    }, f, indent=2, ensure_ascii=False)

print(f' JSON saved to: {output_path}')
print()
print('--- Scores Summary ---')
for k, v in feedback['scores'].items():
    bar = '█' * (v // 5) + '░' * (20 - v // 5)
    print(f'{k.capitalize():12} {bar} {v}/100')

 JSON saved to: C:\Users\mnoum\Downloads\SEMESTER 4\SIT725 - Applied Software Engineering\resume_feedback_output.json

--- Scores Summary ---
Content      ████████████░░░░░░░░ 60/100
Structure    ████████░░░░░░░░░░░░ 40/100
Ats          █████████░░░░░░░░░░░ 45/100
Overall      ██████████░░░░░░░░░░ 50/100
